# So sánh kiến trúc bằng CE, không augmentation

Notebook này huấn luyện các model được chọn bằng Cross-Entropy (CE), không chạy knowledge distillation.

Mỗi model có switch bật/tắt riêng trong ô cấu hình. Tất cả train/validation/test transform chỉ gồm `ToTensor + Normalize`.

## Ablation Stage 4 được cô lập

Notebook giữ nguyên HBCC-Medium CE 84,20% làm baseline và thêm `hbcc_medium_stage4_ablation`. Variant chỉ đổi Stage 4: depth 2, proposal 2×2, `hard_st` với temperature 0.7, hybrid 25% DWConv và channel shuffle. Stage 1–3 vẫn dùng `hard` và được validator khóa chính xác. Resolution `32 → 16 → 8 → 4`, PointReducer 3×3, final feature 288 và classification head đều không đổi.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
import yaml


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'tools' / 'run_ce_experiments.py').is_file():
            return candidate
    raise FileNotFoundError('Khong tim thay repository chua tools/run_ce_experiments.py')


ROOT = find_repo_root()
print('Repository :', ROOT)
print('Python     :', sys.executable)
print('PyTorch    :', torch.__version__)
print('CUDA       :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))

## Cấu hình chạy

- Giữ DATASETS = ['cifar10', 'cifar100'] để chạy đầy đủ cả hai dataset.
- Chỉnh `CE_EPOCHS` để đặt số epoch huấn luyện.
- Đặt switch của model thành `True/False` để bật hoặc tắt model đó.
- Đặt SMOKE = True để kiểm tra nhanh bằng FakeData, mỗi run chỉ một batch.
- FORCE = False bảo vệ các run đã có; run hoàn tất và đúng metadata sẽ tự động được bỏ qua.

In [ ]:
DATASETS = ['cifar10', 'cifar100']
SEEDS = [42]
CE_EPOCHS = 300
DATA_ROOT = ROOT / 'data'
OUTPUT_ROOT = ROOT / 'runs_ce_hbcc_stage4_ablation'
HBCC_ARCHITECTURES = {
    'hbcc_small': 'hbcc_wide_stage4_v1',
    'hbcc_medium': 'hbcc_wide_stage4_v1',
    'hbcc_medium_stage4_ablation': 'hbcc_medium_stage4_ablation_v1',
}
EXPECTED_HBCC_FIELDS = {
    'hbcc_small': {
        'embed_dims': [48, 80, 160, 256],
        'depths': [1, 1, 2, 1],
        'proposals': [[2, 2], [2, 2], [2, 2], [1, 1]],
        'assignment_modes': ['hard', 'hard', 'hard', 'hard'],
        'assignment_temperatures': [1.0, 1.0, 1.0, 1.0],
        'stage_modes': ['hybrid', 'hybrid', 'cluster', 'cluster'],
        'local_branches': ['lbpconv', 'dwconv', 'identity', 'identity'],
        'local_ratios': [0.5, 0.5, 0.0, 0.0],
        'channel_shuffle': [True, True, False, False],
        'stage_drop_path_rates': None,
    },
    'hbcc_medium': {
        'embed_dims': [64, 96, 192, 288],
        'depths': [1, 1, 2, 1],
        'proposals': [[2, 2], [2, 2], [2, 2], [1, 1]],
        'assignment_modes': ['hard', 'hard', 'hard', 'hard'],
        'assignment_temperatures': [1.0, 1.0, 1.0, 1.0],
        'stage_modes': ['hybrid', 'hybrid', 'cluster', 'cluster'],
        'local_branches': ['lbpconv', 'dwconv', 'identity', 'identity'],
        'local_ratios': [0.5, 0.5, 0.0, 0.0],
        'channel_shuffle': [True, True, False, False],
        'stage_drop_path_rates': None,
    },
    'hbcc_medium_stage4_ablation': {
        'embed_dims': [64, 96, 192, 288],
        'depths': [1, 1, 2, 2],
        'proposals': [[2, 2], [2, 2], [2, 2], [2, 2]],
        'assignment_modes': ['hard', 'hard', 'hard', 'hard_st'],
        'assignment_temperatures': [1.0, 1.0, 1.0, 0.7],
        'stage_modes': ['hybrid', 'hybrid', 'cluster', 'hybrid'],
        'local_branches': ['lbpconv', 'dwconv', 'identity', 'dwconv'],
        'local_ratios': [0.5, 0.5, 0.0, 0.25],
        'channel_shuffle': [True, True, False, True],
        'stage_drop_path_rates': [[0.0], [0.02], [0.04, 0.06], [0.08, 0.08]],
    },
}

# ---------- Switch bat/tat tung model ----------
TRAIN_RESNET18 = False
TRAIN_MOBILENET_V2 = False
TRAIN_SHUFFLENET_V2 = False
TRAIN_COC_BASELINE = False
TRAIN_HBCC_SMALL = False
TRAIN_HBCC_MEDIUM_BASELINE = True
TRAIN_HBCC_MEDIUM_STAGE4_ABLATION = True

MODEL_SWITCHES = {
    'resnet18': TRAIN_RESNET18,
    'mobilenet_v2': TRAIN_MOBILENET_V2,
    'shufflenet_v2_x1_0': TRAIN_SHUFFLENET_V2,
    'coc_baseline': TRAIN_COC_BASELINE,
    'hbcc_small': TRAIN_HBCC_SMALL,
    'hbcc_medium': TRAIN_HBCC_MEDIUM_BASELINE,
    'hbcc_medium_stage4_ablation': TRAIN_HBCC_MEDIUM_STAGE4_ABLATION,
}
SELECTED_MODELS = [name for name, enabled in MODEL_SWITCHES.items() if enabled]

SMOKE = False
FORCE = False
SHOW_PROGRESS = True

assert DATASETS and set(DATASETS) <= {'cifar10', 'cifar100'}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert isinstance(CE_EPOCHS, int) and CE_EPOCHS > 0
assert SELECTED_MODELS, 'Phai bat it nhat mot model'
print('Datasets    :', DATASETS)
print('Seeds       :', SEEDS)
print('CE epochs   :', CE_EPOCHS)
print('Models      :', SELECTED_MODELS)
print('Data root   :', DATA_ROOT)
print('Output root :', OUTPUT_ROOT)
print('Smoke       :', SMOKE)

## Preflight

Kiểm tra recipe CE không augmentation, forward shape và toàn bộ invariant của ablation. Preflight sẽ dừng nếu Stage 1–3 của variant lệch baseline, kể cả assignment mode hoặc lịch DropPath.

In [ ]:
RUNNER = ROOT / 'tools' / 'run_ce_experiments.py'


def run_command(command: list[str]) -> None:
    print('\n>', subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


for dataset in DATASETS:
    run_command([
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--epochs', str(CE_EPOCHS),
        '--models', *SELECTED_MODELS,
        '--validate-only',
    ])

## Huấn luyện CE các model đã bật

Mỗi dataset/seed tạo một run CE cho từng model đang bật. Không có teacher checkpoint hoặc loss KD trong runner này.

In [ ]:
for dataset in DATASETS:
    command = [
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--data-root', str(DATA_ROOT),
        '--output', str(OUTPUT_ROOT),
        '--python', sys.executable,
        '--seeds', *[str(seed) for seed in SEEDS],
        '--epochs', str(CE_EPOCHS),
        '--models', *SELECTED_MODELS,
    ]
    if SHOW_PROGRESS:
        command.append('--progress')
    if SMOKE:
        command.append('--smoke')
    if FORCE:
        command.append('--force')
    run_command(command)

## Tổng hợp kết quả test

Cell này đọc `config.yaml` và `test_metrics.json`, chỉ nhận run CE tương thích với cấu hình hiện tại và kiểm tra đủ số model đã bật.

In [ ]:
records = []
selected_datasets = set(DATASETS)
selected_seeds = set(SEEDS)
selected_models = set(SELECTED_MODELS)

for metrics_path in sorted(OUTPUT_ROOT.glob('*/test_metrics.json')):
    run_dir = metrics_path.parent
    config_path = run_dir / 'config.yaml'
    if not config_path.is_file():
        continue
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    dataset = cfg.get('protocol', {}).get('dataset')
    session = cfg.get('protocol', {}).get('session')
    seed = int(cfg.get('train', {}).get('seed', -1))
    run_epochs = int(cfg.get('train', {}).get('epochs', -1))
    is_smoke_run = cfg.get('data', {}).get('name') == 'fake'
    model = cfg.get('experiment', {}).get('model_key')
    expected_epochs = 1 if SMOKE else CE_EPOCHS
    if dataset not in selected_datasets or seed not in selected_seeds:
        continue
    if session != 'baseline' or model not in selected_models:
        continue
    if run_epochs != expected_epochs or is_smoke_run != SMOKE:
        continue
    if cfg.get('train', {}).get('kd_method') != 'none':
        continue
    if model in EXPECTED_HBCC_FIELDS:
        experiment = cfg.get('experiment', {})
        model_cfg = cfg.get('model', {})
        if experiment.get('architecture') != HBCC_ARCHITECTURES[model]:
            continue
        if any(model_cfg.get(field) != value for field, value in EXPECTED_HBCC_FIELDS[model].items()):
            continue
    records.append({
        'dataset': dataset,
        'session': 'ce',
        'model': model,
        'seed': seed,
        'epochs': run_epochs,
        'architecture': cfg.get('experiment', {}).get('architecture'),
        'test_acc1': float(metrics['test_acc1']),
        'test_acc5': metrics.get('test_acc5'),
        'run': run_dir.name,
    })

summary = pd.DataFrame(records)
if summary.empty:
    raise RuntimeError('Khong tim thay ket qua test trong output root')

summary = summary.sort_values(['dataset', 'seed', 'model']).reset_index(drop=True)
baseline_lookup = {
    (row.dataset, int(row.seed)): float(row.test_acc1)
    for row in summary[summary['model'] == 'hbcc_medium'].itertuples()
}
summary['delta_vs_hbcc_medium'] = [
    None if (row.dataset, int(row.seed)) not in baseline_lookup
    else float(row.test_acc1) - baseline_lookup[(row.dataset, int(row.seed))]
    for row in summary.itertuples()
]
expected = len(DATASETS) * len(SEEDS) * len(SELECTED_MODELS)
if len(summary) != expected:
    raise RuntimeError(f'Ket qua chua du: tim thay {len(summary)}/{expected} run')

summary_path = OUTPUT_ROOT / 'ce_summary.csv'
summary.to_csv(summary_path, index=False)
print('Da luu:', summary_path)
summary